In [10]:
# ============================================================
# PHASE 8 — SQL FINANCIAL ANALYSIS
# ============================================================
#
# Purpose:
# Convert the governed Phase 7A analytical models into
# finance / FP&A-ready analytical outputs.
#
# IMPORTANT:
# - Currency is always kept separate.
# - Customer ambiguity is preserved, not hidden.
# - No raw customer-dimension join is used.
# - No invoice rows are multiplied.
# - DSO is NOT presented as formal DSO because the dataset
#   does not contain reliable historical average AR balances.
# - Current AR aging is based on CURRENT_DATE().
# - Scenario outputs are estimates, not forecasts.
#
# Main outputs:
#   1. AR Position
#   2. AR Aging
#   3. Customer Exposure
#   4. Payment Behavior
#   5. Monthly Working-Capital Activity
#   6. Collection-Cycle Analysis
#   7. Cash-Impact Scenario
# ============================================================

import mysql.connector
import pandas as pd
import os
import time
from decimal import Decimal

# ============================================================
# 0. CONFIGURATION
# ============================================================

DB_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE", "working_capital")
}

OUTPUT_DIR = r"C:\Users\hp\Downloads\Working_Capital_ETL"

os.makedirs(OUTPUT_DIR, exist_ok=True)

start_time = time.time()

conn = mysql.connector.connect(**DB_CONFIG)

print("=" * 70)
print("PHASE 8 — SQL FINANCIAL ANALYSIS")
print("=" * 70)

print("MYSQL connection: PASS")


# ============================================================
# 1. VALIDATE PHASE 7A SOURCE
# ============================================================

print("\n1. VALIDATE PHASE 7A SOURCE")
print("-" * 70)

cur = conn.cursor()

cur.execute("""
    SELECT COUNT(*)
    FROM analytics.ae_ar_invoice
""")

ar_invoice_rows = cur.fetchone()[0]

cur.execute("""
    SELECT COUNT(DISTINCT invoice_id)
    FROM analytics.ae_ar_invoice
""")

ar_invoice_ids = cur.fetchone()[0]

print(f"AE AR invoice rows   : {ar_invoice_rows:,}")
print(f"Unique invoice IDs   : {ar_invoice_ids:,}")

if ar_invoice_rows != 12000:
    raise ValueError(
        f"Expected 12,000 AE AR invoice rows, got {ar_invoice_rows:,}"
    )

if ar_invoice_ids != 12000:
    raise ValueError(
        f"Expected 12,000 unique invoice IDs, got {ar_invoice_ids:,}"
    )

print("Phase 7A source validation: PASS")


# ============================================================
# 2. VALIDATE ACTUAL CUSTOMER-QUALITY STRUCTURE
# ============================================================

print("\n2. VALIDATE PHASE 7A CUSTOMER-QUALITY STRUCTURE")
print("-" * 70)

cur.execute("""
    SELECT COLUMN_NAME
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'analytics'
      AND TABLE_NAME = 'ae_ar_invoice'
""")

ae_ar_columns = {row[0] for row in cur.fetchall()}

required_ar_columns = {
    "customer_id",
    "customer_dimension_row_count",
    "customer_distinct_name_count",
    "customer_distinct_region_count",
    "customer_dimension_status"
}

missing_ar_columns = required_ar_columns - ae_ar_columns

if missing_ar_columns:
    print("Missing required columns:", sorted(missing_ar_columns))
    raise ValueError(
        "AE AR invoice customer-quality structure is incomplete."
    )

print("Required customer-quality columns: PASS")


# ============================================================
# 3. RESET PHASE 8 TABLES
# ============================================================

print("\n3. RESET PHASE 8 TABLES")
print("-" * 70)

phase8_tables = [
    "fin_cash_impact_scenario",
    "fin_dso_analysis",
    "fin_monthly_working_capital",
    "fin_payment_behavior",
    "fin_customer_exposure",
    "fin_ar_aging",
    "fin_ar_position"
]

for table in phase8_tables:
    cur.execute(
        f"DROP TABLE IF EXISTS analytics.{table}"
    )

conn.commit()

print("Phase 8 tables reset: PASS")


# ============================================================
# 4. BUILD AR POSITION
# ============================================================
#
# Grain:
#   1 row per currency
#
# Business meaning:
#   How much invoiced value, collected value and outstanding
#   value exists by currency.
#
# IMPORTANT:
#   MAD and EUR are NEVER combined.
#
# Financial eligibility:
#   FULL / PARTIAL
#
# Rows classified as NOT_USABLE_FOR_AMOUNT_ANALYSIS are excluded
# from financial amount calculations.
# ============================================================

print("\n4. BUILD AR POSITION")
print("-" * 70)

cur.execute("""
    CREATE TABLE analytics.fin_ar_position AS

    SELECT
        currency,

        COUNT(*) AS invoice_count,

        SUM(invoice_amount) AS invoice_amount,

        SUM(COALESCE(payment_amount, 0)) AS payment_amount,

        SUM(
            invoice_amount
            - COALESCE(payment_amount, 0)
        ) AS outstanding_amount,

        SUM(
            CASE
                WHEN status = 'Open'
                THEN invoice_amount - COALESCE(payment_amount, 0)
                ELSE 0
            END
        ) AS open_outstanding_amount,

        SUM(
            CASE
                WHEN is_paid_late = 1
                THEN invoice_amount - COALESCE(payment_amount, 0)
                ELSE 0
            END
        ) AS late_payment_outstanding_amount

    FROM analytics.ae_ar_invoice

    WHERE financial_analytical_eligibility
          IN ('FULL', 'PARTIAL')

    GROUP BY currency
""")

conn.commit()

print("AR position: CREATED")


# ============================================================
# 5. BUILD AR AGING
# ============================================================
#
# Grain:
#   currency + aging bucket
#
# This is CURRENT aging, not historical month-end aging.
#
# Because the current system date is used, the output answers:
#
#   "How old is the currently open receivable balance today?"
#
# ============================================================

print("\n5. BUILD AR AGING")
print("-" * 70)

cur.execute("""
    CREATE TABLE analytics.fin_ar_aging AS

    SELECT
        currency,

        CASE

            WHEN DATEDIFF(CURRENT_DATE(), due_date) <= 0
                THEN 'Current'

            WHEN DATEDIFF(CURRENT_DATE(), due_date) BETWEEN 1 AND 30
                THEN '1-30 Days'

            WHEN DATEDIFF(CURRENT_DATE(), due_date) BETWEEN 31 AND 60
                THEN '31-60 Days'

            WHEN DATEDIFF(CURRENT_DATE(), due_date) BETWEEN 61 AND 90
                THEN '61-90 Days'

            ELSE '90+ Days'

        END AS aging_bucket,

        COUNT(*) AS invoice_count,

        SUM(
            invoice_amount
            - COALESCE(payment_amount, 0)
        ) AS outstanding_amount

    FROM analytics.ae_ar_invoice

    WHERE status = 'Open'

      AND financial_analytical_eligibility
          IN ('FULL', 'PARTIAL')

    GROUP BY
        currency,

        CASE

            WHEN DATEDIFF(CURRENT_DATE(), due_date) <= 0
                THEN 'Current'

            WHEN DATEDIFF(CURRENT_DATE(), due_date) BETWEEN 1 AND 30
                THEN '1-30 Days'

            WHEN DATEDIFF(CURRENT_DATE(), due_date) BETWEEN 31 AND 60
                THEN '31-60 Days'

            WHEN DATEDIFF(CURRENT_DATE(), due_date) BETWEEN 61 AND 90
                THEN '61-90 Days'

            ELSE '90+ Days'

        END
""")

conn.commit()

print("AR aging: CREATED")


# ============================================================
# 6. BUILD CUSTOMER EXPOSURE
# ============================================================
#
# Grain:
#   currency + customer_id
#
# CRITICAL:
#   We DO NOT expose customer_name or region as a single value.
#
# Why?
#
#   The project demonstrated that customer_id is ambiguous.
#
#   Example:
#       C0001
#       56 dimension rows
#       8 names
#       7 regions
#
# Selecting MAX(customer_name) would create false certainty.
#
# Instead, we expose the ambiguity itself.
# ============================================================

print("\n6. BUILD CUSTOMER EXPOSURE")
print("-" * 70)

cur.execute("""
    CREATE TABLE analytics.fin_customer_exposure AS

    SELECT

        currency,

        customer_id,

        MAX(customer_dimension_status)
            AS customer_dimension_status,

        MAX(customer_dimension_row_count)
            AS customer_dimension_row_count,

        MAX(customer_distinct_name_count)
            AS distinct_customer_names,

        MAX(customer_distinct_region_count)
            AS distinct_regions,

        COUNT(*) AS invoice_count,

        SUM(invoice_amount) AS invoice_amount,

        SUM(COALESCE(payment_amount, 0))
            AS payment_amount,

        SUM(
            invoice_amount
            - COALESCE(payment_amount, 0)
        ) AS outstanding_amount,

        SUM(
            CASE
                WHEN currently_overdue = 1
                THEN invoice_amount
                     - COALESCE(payment_amount, 0)
                ELSE 0
            END
        ) AS overdue_amount

    FROM analytics.ae_ar_invoice

    WHERE financial_analytical_eligibility
          IN ('FULL', 'PARTIAL')

    GROUP BY
        currency,
        customer_id
""")

conn.commit()

print("Customer exposure: CREATED")


# ============================================================
# 7. BUILD PAYMENT BEHAVIOR
# ============================================================
#
# Grain:
#   currency + payment_method + status
#
# Purpose:
#   Understand collection behavior by payment channel and
#   invoice status.
# ============================================================

print("\n7. BUILD PAYMENT BEHAVIOR")
print("-" * 70)

cur.execute("""
    CREATE TABLE analytics.fin_payment_behavior AS

    SELECT

        currency,

        payment_method,

        status,

        COUNT(*) AS invoice_count,

        SUM(invoice_amount) AS invoice_amount,

        SUM(COALESCE(payment_amount, 0))
            AS payment_amount,

        AVG(
            CASE
                WHEN status = 'Paid'
                THEN days_to_payment
            END
        ) AS avg_days_to_payment,

        AVG(
            CASE
                WHEN status = 'Paid'
                THEN days_from_due_date
            END
        ) AS avg_days_from_due_date,

        SUM(
            CASE
                WHEN is_paid_late = 1
                THEN 1
                ELSE 0
            END
        ) AS late_payment_count

    FROM analytics.ae_ar_invoice

    WHERE financial_analytical_eligibility
          IN ('FULL', 'PARTIAL')

    GROUP BY
        currency,
        payment_method,
        status
""")

conn.commit()

print("Payment behavior: CREATED")


# ============================================================
# 8. BUILD MONTHLY WORKING-CAPITAL ACTIVITY
# ============================================================
#
# Grain:
#   currency + invoice month
#
# IMPORTANT:
# This is TRANSACTION ACTIVITY.
#
# It is NOT historical month-end AR balance.
#
# Therefore it should not be interpreted as:
#
#   "AR balance at month-end"
#
# Instead it answers:
#
#   "What invoicing and payment activity occurred during
#    each month?"
# ============================================================

print("\n8. BUILD MONTHLY WORKING-CAPITAL ACTIVITY")
print("-" * 70)

cur.execute("""
    CREATE TABLE analytics.fin_monthly_working_capital AS

    SELECT

        currency,

        DATE_FORMAT(
            invoice_date,
            '%Y-%m-01'
        ) AS invoice_month,

        COUNT(*) AS invoice_count,

        SUM(invoice_amount) AS invoice_amount,

        SUM(COALESCE(payment_amount, 0))
            AS payment_amount,

        AVG(
            CASE
                WHEN status = 'Paid'
                THEN days_to_payment
            END
        ) AS avg_days_to_payment,

        SUM(
            CASE
                WHEN is_paid_late = 1
                THEN 1
                ELSE 0
            END
        ) AS late_payment_count

    FROM analytics.ae_ar_invoice

    WHERE financial_analytical_eligibility
          IN ('FULL', 'PARTIAL')

    GROUP BY
        currency,
        DATE_FORMAT(invoice_date, '%Y-%m-01')
""")

conn.commit()

print("Monthly working-capital activity: CREATED")


# ============================================================
# 9. BUILD COLLECTION-CYCLE ANALYSIS
# ============================================================
#
# We deliberately DO NOT call this formal DSO.
#
# Proper DSO normally requires:
#
#     Average AR / Credit Sales × Number of Days
#
# The dataset does not provide reliable historical AR balances.
#
# Instead, this model measures observed invoice-to-payment
# collection behavior.
# ============================================================

print("\n9. BUILD COLLECTION-CYCLE ANALYSIS")
print("-" * 70)

cur.execute("""
    CREATE TABLE analytics.fin_dso_analysis AS

    SELECT

        currency,

        COUNT(*) AS paid_invoice_count,

        AVG(days_to_payment)
            AS avg_collection_days,

        MIN(days_to_payment)
            AS min_collection_days,

        MAX(days_to_payment)
            AS max_collection_days,

        AVG(
            CASE
                WHEN days_from_due_date > 0
                THEN days_from_due_date
            END
        ) AS avg_days_late_after_due_date,

        SUM(
            CASE
                WHEN is_paid_late = 1
                THEN 1
                ELSE 0
            END
        ) AS late_paid_invoice_count,

        ROUND(
            100.0
            * SUM(
                CASE
                    WHEN is_paid_late = 1
                    THEN 1
                    ELSE 0
                END
              )
            / NULLIF(COUNT(*), 0),
            2
        ) AS late_payment_rate_pct

    FROM analytics.ae_ar_invoice

    WHERE status = 'Paid'

      AND financial_analytical_eligibility
          IN ('FULL', 'PARTIAL')

      AND payment_date IS NOT NULL

    GROUP BY currency
""")

conn.commit()

print("Collection-cycle analysis: CREATED")


# ============================================================
# 10. BUILD CASH-IMPACT SCENARIO
# ============================================================
#
# Scenario:
#
#   "What if collection time improves by 5 days?"
#
# This is an ESTIMATE.
# It is NOT a forecast.
#
# Approximation:
#
#   Open AR × (5 / observed average collection days)
#
# This gives an indicative cash-release amount.
#
# Currency is kept separate.
# ============================================================

print("\n10. BUILD CASH-IMPACT SCENARIO")
print("-" * 70)

cur.execute("""
    CREATE TABLE analytics.fin_cash_impact_scenario AS

    SELECT

        ar.currency,

        ar.open_outstanding_amount,

        d.avg_collection_days,

        5 AS assumed_improvement_days,

        ROUND(
            ar.open_outstanding_amount
            * (
                5.0
                / NULLIF(d.avg_collection_days, 0)
              ),
            2
        ) AS estimated_cash_release

    FROM analytics.fin_ar_position ar

    INNER JOIN analytics.fin_dso_analysis d
        ON ar.currency = d.currency
""")

conn.commit()

print("Cash-impact scenario: CREATED")


# ============================================================
# 11. FINANCIAL RECONCILIATION
# ============================================================
#
# Verify that Phase 8 did not alter financial totals.
# ============================================================

print("\n11. FINANCIAL RECONCILIATION")
print("-" * 70)

cur.execute("""
    SELECT
        currency,
        SUM(invoice_amount) AS invoice_amount,
        SUM(COALESCE(payment_amount, 0)) AS payment_amount
    FROM analytics.ae_ar_invoice
    WHERE financial_analytical_eligibility
          IN ('FULL', 'PARTIAL')
    GROUP BY currency
    ORDER BY currency
""")

source_financial = cur.fetchall()

cur.execute("""
    SELECT
        currency,
        invoice_amount,
        payment_amount
    FROM analytics.fin_ar_position
    ORDER BY currency
""")

position_financial = cur.fetchall()

source_dict = {
    row[0]: (
        Decimal(str(row[1])),
        Decimal(str(row[2]))
    )
    for row in source_financial
}

position_dict = {
    row[0]: (
        Decimal(str(row[1])),
        Decimal(str(row[2]))
    )
    for row in position_financial
}

financial_reconciliation_pass = True

for currency in source_dict:

    source_invoice, source_payment = source_dict[currency]

    position_invoice, position_payment = position_dict.get(
        currency,
        (Decimal("0"), Decimal("0"))
    )

    invoice_match = (
        source_invoice == position_invoice
    )

    payment_match = (
        source_payment == position_payment
    )

    print(
        f"{currency}: "
        f"invoice={'PASS' if invoice_match else 'FAIL'}, "
        f"payment={'PASS' if payment_match else 'FAIL'}"
    )

    if not invoice_match or not payment_match:
        financial_reconciliation_pass = False

if not financial_reconciliation_pass:
    raise ValueError(
        "Phase 8 financial reconciliation failed."
    )

print("Financial reconciliation: PASS")


# ============================================================
# 12. VALIDATE AR POSITION
# ============================================================

print("\n12. VALIDATE AR POSITION")
print("-" * 70)

cur.execute("""
    SELECT COUNT(*)
    FROM analytics.fin_ar_position
""")

ar_position_rows = cur.fetchone()[0]

cur.execute("""
    SELECT COUNT(*)
    FROM analytics.fin_ar_position
    WHERE outstanding_amount < 0
""")

negative_ar_rows = cur.fetchone()[0]

print(f"AR position rows: {ar_position_rows}")
print(f"Negative aggregate outstanding rows: {negative_ar_rows}")

if negative_ar_rows != 0:
    print(
        "WARNING: negative aggregate outstanding exists. "
        "This reflects underlying payment/invoice anomalies "
        "and should be investigated rather than silently capped."
    )
else:
    print("Negative aggregate outstanding: PASS")


# ============================================================
# 13. VALIDATE AR AGING RECONCILIATION
# ============================================================
#
# Aging total should equal open outstanding by currency.
# ============================================================

print("\n13. VALIDATE AR AGING RECONCILIATION")
print("-" * 70)

cur.execute("""
    SELECT
        currency,
        SUM(outstanding_amount)
    FROM analytics.fin_ar_aging
    GROUP BY currency
    ORDER BY currency
""")

aging_totals = {
    row[0]: Decimal(str(row[1]))
    for row in cur.fetchall()
}

cur.execute("""
    SELECT
        currency,
        open_outstanding_amount
    FROM analytics.fin_ar_position
    ORDER BY currency
""")

open_totals = {
    row[0]: Decimal(str(row[1]))
    for row in cur.fetchall()
}

aging_pass = True

for currency in open_totals:

    aging_value = aging_totals.get(
        currency,
        Decimal("0")
    )

    open_value = open_totals[currency]

    match = aging_value == open_value

    print(
        f"{currency}: "
        f"{'PASS' if match else 'FAIL'}"
    )

    if not match:
        aging_pass = False

if not aging_pass:
    raise ValueError(
        "AR aging reconciliation failed."
    )

print("AR aging reconciliation: PASS")


# ============================================================
# 14. VALIDATE CUSTOMER EXPOSURE GRAIN
# ============================================================

print("\n14. VALIDATE CUSTOMER EXPOSURE")
print("-" * 70)

cur.execute("""
    SELECT
        currency,
        customer_id,
        COUNT(*) AS row_count
    FROM analytics.fin_customer_exposure
    GROUP BY
        currency,
        customer_id
    HAVING COUNT(*) <> 1
""")

duplicate_customer_grain = cur.fetchall()

if duplicate_customer_grain:
    raise ValueError(
        "Customer exposure grain violation detected."
    )

print(
    "Customer exposure grain "
    "(currency + customer_id): PASS"
)


# ============================================================
# 15. EXPORT PHASE 8 OUTPUTS
# ============================================================

print("\n15. EXPORT PHASE 8 OUTPUTS")
print("-" * 70)

output_tables = {
    "fin_ar_position":
        "Phase8_AR_Position.csv",

    "fin_ar_aging":
        "Phase8_AR_Aging.csv",

    "fin_customer_exposure":
        "Phase8_Customer_Exposure.csv",

    "fin_payment_behavior":
        "Phase8_Payment_Behavior.csv",

    "fin_monthly_working_capital":
        "Phase8_Monthly_Working_Capital.csv",

    "fin_dso_analysis":
        "Phase8_Collection_Cycle.csv",

    "fin_cash_impact_scenario":
        "Phase8_Cash_Impact_Scenario.csv"
}

for table_name, file_name in output_tables.items():

    df = pd.read_sql(
        f"SELECT * FROM analytics.{table_name}",
        conn
    )

    output_path = os.path.join(
        OUTPUT_DIR,
        file_name
    )

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"{file_name:<40} "
        f"{len(df):>8,} rows"
    )


# ============================================================
# 16. MANAGEMENT SUMMARY
# ============================================================

print("\n16. MANAGEMENT SUMMARY")
print("-" * 70)

print("\nAR POSITION")

df_ar = pd.read_sql(
    """
    SELECT *
    FROM analytics.fin_ar_position
    ORDER BY currency
    """,
    conn
)

print(df_ar.to_string(index=False))


print("\nAR AGING")

df_aging = pd.read_sql(
    """
    SELECT *
    FROM analytics.fin_ar_aging
    ORDER BY
        currency,
        FIELD(
            aging_bucket,
            'Current',
            '1-30 Days',
            '31-60 Days',
            '61-90 Days',
            '90+ Days'
        )
    """,
    conn
)

print(df_aging.to_string(index=False))


print("\nCOLLECTION CYCLE")

df_collection = pd.read_sql(
    """
    SELECT *
    FROM analytics.fin_dso_analysis
    ORDER BY currency
    """,
    conn
)

print(df_collection.to_string(index=False))


print("\nCASH-IMPACT SCENARIO")

df_scenario = pd.read_sql(
    """
    SELECT *
    FROM analytics.fin_cash_impact_scenario
    ORDER BY currency
    """,
    conn
)

print(df_scenario.to_string(index=False))


# ============================================================
# 17. FINAL PHASE 8 STATUS
# ============================================================

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("PHASE 8 FINAL STATUS")
print("=" * 70)

print("Source validation             : PASS")
print("Customer-quality structure    : PASS")
print("AR position                   : CREATED")
print("AR aging                      : CREATED")
print("Customer exposure             : CREATED")
print("Payment behavior              : CREATED")
print("Monthly WC activity           : CREATED")
print("Collection-cycle analysis     : CREATED")
print("Cash-impact scenario          : CREATED")
print("Financial reconciliation      : PASS")
print("AR aging reconciliation       : PASS")
print("Customer exposure grain       : PASS")

print(f"\nElapsed time: {elapsed:.2f} seconds")

print("\nPhase 8 completed successfully.")

cur.close()
conn.close()

PHASE 8 — SQL FINANCIAL ANALYSIS
MYSQL connection: PASS

1. VALIDATE PHASE 7A SOURCE
----------------------------------------------------------------------
AE AR invoice rows   : 12,000
Unique invoice IDs   : 12,000
Phase 7A source validation: PASS

2. VALIDATE PHASE 7A CUSTOMER-QUALITY STRUCTURE
----------------------------------------------------------------------
Required customer-quality columns: PASS

3. RESET PHASE 8 TABLES
----------------------------------------------------------------------
Phase 8 tables reset: PASS

4. BUILD AR POSITION
----------------------------------------------------------------------
AR position: CREATED

5. BUILD AR AGING
----------------------------------------------------------------------
AR aging: CREATED

6. BUILD CUSTOMER EXPOSURE
----------------------------------------------------------------------
Customer exposure: CREATED

7. BUILD PAYMENT BEHAVIOR
----------------------------------------------------------------------
Payment behavior: CREA

C:\Users\hp\AppData\Local\Temp\ipykernel_18080\3245124278.py:935: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\hp\AppData\Local\Temp\ipykernel_18080\3245124278.py:965: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ar = pd.read_sql(
C:\Users\hp\AppData\Local\Temp\ipykernel_18080\3245124278.py:979: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_aging = pd.read_sql(


Phase8_AR_Position.csv                          2 rows
Phase8_AR_Aging.csv                             2 rows
Phase8_Customer_Exposure.csv                  162 rows
Phase8_Payment_Behavior.csv                    19 rows
Phase8_Monthly_Working_Capital.csv             50 rows
Phase8_Collection_Cycle.csv                     2 rows
Phase8_Cash_Impact_Scenario.csv                 2 rows

16. MANAGEMENT SUMMARY
----------------------------------------------------------------------

AR POSITION
currency  invoice_count  invoice_amount  payment_amount  outstanding_amount  open_outstanding_amount  late_payment_outstanding_amount
     EUR            601      4802482.14      4080652.03           721830.11                735424.33                         61004.04
     MAD          11289     87184647.40     71883670.76         15300976.64              14789868.43                        413948.71

AR AGING
currency aging_bucket  invoice_count  outstanding_amount
     EUR     90+ Days            102  

C:\Users\hp\AppData\Local\Temp\ipykernel_18080\3245124278.py:1002: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_collection = pd.read_sql(
C:\Users\hp\AppData\Local\Temp\ipykernel_18080\3245124278.py:1016: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_scenario = pd.read_sql(
